# Treinamento CNN Pontinhos — V8 (Canais Estruturais)

Baseado em `Treinamento_CNN_Arena_Sagaz_V7_Sample_Weight.ipynb`.

**Mudancas em relacao ao V7:**
- Lambda `para_grid_de_caixas` **removida** — a CNN recebe diretamente o tensor `canais (N,4,3,K)` pre-computado nos NPZ da Fase A.2.
- Input shape: `(4, 3, K)` onde K = `len(CANAIS_TREINAMENTO)` (configuravel).
- Sem normalizacao extra: os canais ja sao `{0,1} int8` → apenas `.astype(float32)`.
- OMA corrigido: aceita qualquer jogada com Q-value maximo (empates incluidos).
- Metricas por canal: Tabela 1 (amostras por canal/fase), Tabela 2 (Top-1/3/5/OMA por canal), Correlacao canal x erro.

**Treino**: Google Colab + Google Drive. Le NPZs da Fase A.2 (`dados/profundidade_minimax_11_v7_adaptativo/`).


In [ ]:
# =========================================================================
# CONFIGURACAO — edite aqui antes de rodar
# =========================================================================
from google.colab import drive
drive.mount('/content/drive')

# Pasta no Drive onde o .tflite sera salvo
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/Arena Sagaz/CNN'

# Pasta local dos NPZ (Fase A.2 — com `canais` e `nomes_canais`)
# Colab com upload direto (arquivos ao lado de drive/ e sample_data/): PASTA_NPZ = '/content'
# Colab via Google Drive: PASTA_NPZ = '/content/drive/MyDrive/Arena Sagaz/dados/profundidade_minimax_11_v7_adaptativo'
# Local (repo clone): PASTA_NPZ = '../dados/profundidade_minimax_11_v7_adaptativo'
PASTA_NPZ = '/content'

# Canais a incluir no treinamento (subconjunto dos 11 canonicos da Fase A.2).
# Use None para incluir todos os 11. A ordem define os indices no tensor de entrada.
CANAIS_TREINAMENTO = [
    'aresta_topo', 'aresta_base', 'aresta_esquerda', 'aresta_direita',
    'caixa_fechada', 'eh_grau3', 'eh_grau2', 'em_cadeia_curta',
    'em_cadeia_longa', 'em_loop', 'em_cadeia_aberta_uma_ponta',
]

# Para validacao Fase B (baseline 5 canais), substitua pela linha abaixo:
# CANAIS_TREINAMENTO = ['aresta_topo', 'aresta_base', 'aresta_esquerda', 'aresta_direita', 'caixa_fechada']

# 'INCLUI_DUPLICADAS' (todas as amostras) ou 'DISTINTAS' (apenas matrizes unicas)
UTILIZACAO_MATRIZES = 'INCLUI_DUPLICADAS'

# USE_SAMPLE_WEIGHT = True: aplica peso por qtd_tracos (raro/frequente). False = sem peso.
USE_SAMPLE_WEIGHT = False

import os
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print(f"Drive montado. Modelos serao salvos em: {DRIVE_OUTPUT_DIR}")
print(f"Lendo NPZ de: {PASTA_NPZ}")
print(f"Canais selecionados ({len(CANAIS_TREINAMENTO)}): {CANAIS_TREINAMENTO}")

In [ ]:
import os
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
sns.set_theme(style="whitegrid")
np.random.seed(42)
tf.keras.utils.set_random_seed(42)

print("TensorFlow:", tf.__version__)
print("GPU disponivel:", len(tf.config.list_physical_devices('GPU')) > 0)


In [ ]:
# =========================================================================
# 1. LEITURA DOS DADOS
# =========================================================================

# Ordem canonica dos 11 canais (espelha NOMES_CANAIS em analisador_estrutural_pontinhos.py)
NOMES_CANAIS_REF = (
    'aresta_topo', 'aresta_base', 'aresta_esquerda', 'aresta_direita',
    'caixa_fechada', 'eh_grau3', 'eh_grau2', 'em_cadeia_curta',
    'em_cadeia_longa', 'em_loop', 'em_cadeia_aberta_uma_ponta',
)

# Resolve indices dos canais selecionados dentro do tensor (N, 4, 3, 11)
_canais_alvo = CANAIS_TREINAMENTO if CANAIS_TREINAMENTO is not None else list(NOMES_CANAIS_REF)
INDICES_CANAIS = [NOMES_CANAIS_REF.index(c) for c in _canais_alvo]
K = len(INDICES_CANAIS)
print(f"Canais para treino ({K}): {_canais_alvo}")
print(f"Indices no tensor (N,4,3,11): {INDICES_CANAIS}")

# Mapeamento fator multiplicador (Amostras Brutas / Amostras Distintas)
TRACOS_WEIGHTS = {
    1: 815.74,  2: 54.38,   3: 5.65,   4: 1.47,   5: 1.08,
    6: 1.02,    7: 1.01,    8: 1.00,   9: 1.00,   10: 1.00,
    11: 1.00,   12: 1.00,   13: 1.00,  14: 1.00,  15: 1.00,
    16: 1.00,   17: 1.00,   18: 1.01,  19: 1.03,  20: 1.07,
    21: 1.17,   22: 1.37,   23: 1.70,  24: 2.34,  25: 3.51,
    26: 6.32,   27: 14.58,  28: 41.05, 29: 150.52, 30: 815.74,
}

# =========================================================================
# 1.1 LEITURA DOS LOTES
# =========================================================================
arquivos_npz = sorted(glob.glob(os.path.join(PASTA_NPZ, '*.npz')))
print(f'\nEncontrados {len(arquivos_npz)} arquivos NPZ.')

lista_estados, lista_canais = [], []
lista_rotulos, lista_scores, lista_qtd_tracos = [], [], []
labels_canonicos = None

for arquivo in arquivos_npz:
    dados = np.load(arquivo, allow_pickle=True)
    lista_estados.append(dados['estados'])            # para dedup DISTINTAS
    lista_canais.append(dados['canais'])              # (N, 4, 3, 11) int8
    lista_rotulos.append(dados['melhor_jogada'])      # (N,) <U5
    lista_scores.append(dados['score_melhor_jogada']) # (N, 31) float32
    lista_qtd_tracos.append(dados['qtd_tracos'])      # (N,) int8
    if labels_canonicos is None:
        labels_canonicos = dados['labels_canonicos'].tolist()

estados_raw    = np.concatenate(lista_estados,    axis=0)
canais_raw     = np.concatenate(lista_canais,     axis=0)  # (N, 4, 3, 11) int8
y_str          = np.concatenate(lista_rotulos,    axis=0)
scores_raw     = np.concatenate(lista_scores,     axis=0)  # (N, 31) float32
qtd_tracos_all = np.concatenate(lista_qtd_tracos, axis=0).astype(np.int8)

print(f'Total de amostras brutas: {len(canais_raw):,}')

# =========================================================================
# 1.2 FILTRAGEM DE DUPLICATAS (opcional)
# =========================================================================
if UTILIZACAO_MATRIZES == 'DISTINTAS':
    print('Filtrando apenas matrizes distintas...')
    _, unique_idx = np.unique(estados_raw.reshape(len(estados_raw), -1), axis=0, return_index=True)
    unique_idx.sort()
    canais_raw     = canais_raw[unique_idx]
    y_str          = y_str[unique_idx]
    scores_raw     = scores_raw[unique_idx]
    qtd_tracos_all = qtd_tracos_all[unique_idx]
    print(f'Total apos filtragem (DISTINTAS): {len(canais_raw):,}')
else:
    print('Utilizando todas as matrizes (INCLUI_DUPLICADAS).')
del estados_raw  # libera memoria

# =========================================================================
# 1.3 TENSOR DE ENTRADA
# V8: sem normalizacao — canais ja sao {0,1} int8 → float32 direto.
# =========================================================================
X = canais_raw[:, :, :, INDICES_CANAIS].astype(np.float32)  # (N, 4, 3, K) {0.0, 1.0}
print(f'Shape entrada: {X.shape} | dtype: {X.dtype}')
assert set(X.flat) - {0.0, 1.0} == set(), "ERRO: valores fora de {0,1} detectados!"

# =========================================================================
# 1.4 SOFT TARGETS (KL Divergence) — identico ao V7
# Usa score_melhor_jogada (supervisao Minimax profunda, nao score_jogada).
# =========================================================================
SCORE_IND = -1e9
label_to_idx       = {l: i for i, l in enumerate(labels_canonicos)}
num_classes        = len(labels_canonicos)
indice_para_rotulo = {i: l for i, l in enumerate(labels_canonicos)}
T = 1.0  # temperatura

y_soft = np.zeros((len(scores_raw), num_classes), dtype=np.float32)
for i, sv in enumerate(scores_raw):
    mask = sv > SCORE_IND
    if mask.sum() == 0:
        y_soft[i] = 1.0 / num_classes
        continue
    vals = sv[mask] / T
    vals -= vals.max()
    exp_v = np.exp(vals)
    y_soft[i, mask] = exp_v / exp_v.sum()

y_idx = y_soft.argmax(axis=1)  # argmax do soft target — para Top-1 do Keras

# =========================================================================
# 1.5 FASE DO JOGO — usa qtd_tracos do NPZ diretamente
# V8 nao recomputa da matriz (nao carregamos estados para esta finalidade).
# =========================================================================
fase_jogo = np.digitize(qtd_tracos_all, bins=[12, 18, 24, 29]).astype(np.int8)
FASE_NAMES = {
    0: 'Abertura (0-11)',   1: '1a Metade (12-17)',
    2: '2a Metade (18-23)', 3: 'Fase Quente (24-28)', 4: 'Final (29-31)',
}
print('\nDistribuicao por fase do jogo:')
for f in sorted(set(fase_jogo.tolist())):
    n = (fase_jogo == f).sum()
    print(f'  {FASE_NAMES.get(f, str(f))}: {n:,} ({n/len(fase_jogo)*100:.1f}%)')

# =========================================================================
# 1.6 SPLIT 70/15/15 — estratificado por fase do jogo (identico ao V7)
# =========================================================================
idx_all = np.arange(len(X))
idx_tv, idx_test = train_test_split(
    idx_all, test_size=0.15, random_state=42, stratify=fase_jogo)
idx_train, idx_val = train_test_split(
    idx_tv, test_size=0.15/0.85, random_state=42, stratify=fase_jogo[idx_tv])

X_train, y_train = X[idx_train], y_soft[idx_train]
X_val,   y_val   = X[idx_val],   y_soft[idx_val]
X_test,  y_test  = X[idx_test],  y_soft[idx_test]

y_test_idx  = y_idx[idx_test]          # argmax do soft target (referencia Top-1 Keras)
S_test      = scores_raw[idx_test]     # Q-values brutos — para OMA correto
fase_test   = fase_jogo[idx_test]
tracos_test = qtd_tracos_all[idx_test]
canais_test = canais_raw[idx_test]     # (N_test, 4, 3, 11) — todos 11 para analise

# =========================================================================
# 1.7 SAMPLE WEIGHT (opcional) — identico ao V7, clip 20.0
# =========================================================================
if USE_SAMPLE_WEIGHT:
    qtd_treino = qtd_tracos_all[idx_train]
    sw_raw = np.array([TRACOS_WEIGHTS.get(int(t), 1.0) for t in qtd_treino], dtype=np.float32)
    sw = np.clip(sw_raw, 0.0, 20.0)
    print('sample_weight ativado (CLIP 20.0)')
else:
    sw = None

print(f'\nTreino: {len(X_train):,}  |  Val: {len(X_val):,}  |  Teste: {len(X_test):,}')
print(f'X_train shape: {X_train.shape}')

_dist = pd.DataFrame({
    'Treino': pd.Series(fase_jogo[idx_train]).value_counts(normalize=True).sort_index() * 100,
    'Val':    pd.Series(fase_jogo[idx_val]).value_counts(normalize=True).sort_index() * 100,
    'Teste':  pd.Series(fase_jogo[idx_test]).value_counts(normalize=True).sort_index() * 100,
}).round(2)
_dist.index = [FASE_NAMES[i] for i in _dist.index]
print('\nDistribuicao por fase nos tres conjuntos (%):')
print(_dist.to_string())


In [ ]:
# =========================================================================
# 2. ARQUITETURA BoxNet v3 — V8 (sem Lambda, input (4,3,K))
# Identica ao V7 exceto: input shape (4,3,K) em vez de (9,7,1),
# sem camada Lambda para_grid_de_caixas.
# =========================================================================

def bloco_residual_separavel(x, filtros, l2=2e-4, dropout=0.15):
    atalho = x
    y = layers.SeparableConv2D(
        filtros, (3, 3), padding='same', use_bias=False,
        depthwise_regularizer=regularizers.l2(l2),
        pointwise_regularizer=regularizers.l2(l2),
    )(x)
    y = layers.BatchNormalization()(y)
    y = layers.Activation('relu')(y)
    y = layers.SpatialDropout2D(dropout)(y)
    y = layers.SeparableConv2D(
        filtros, (3, 3), padding='same', use_bias=False,
        depthwise_regularizer=regularizers.l2(l2),
        pointwise_regularizer=regularizers.l2(l2),
    )(y)
    y = layers.BatchNormalization()(y)
    if atalho.shape[-1] != filtros:
        atalho = layers.Conv2D(filtros, (1, 1), padding='same', use_bias=False,
                               kernel_regularizer=regularizers.l2(l2))(atalho)
        atalho = layers.BatchNormalization()(atalho)
    out = layers.Add()([y, atalho])
    out = layers.Activation('relu')(out)
    return out


L2 = 2e-4
INPUT_SHAPE = (4, 3, K)

inputs = Input(shape=INPUT_SHAPE, name='canais_estruturais')

# V8: sem Lambda — recebe (B, 4, 3, K) diretamente do NPZ.
x = layers.Conv2D(32, (3, 3), padding='same', use_bias=False,
                  kernel_regularizer=regularizers.l2(L2))(inputs)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)

x = bloco_residual_separavel(x, 32, l2=L2, dropout=0.15)
x = bloco_residual_separavel(x, 48, l2=L2, dropout=0.20)

gap  = layers.GlobalAveragePooling2D()(x)
flat = layers.Flatten()(x)
h    = layers.Concatenate()([gap, flat])

h    = layers.Dense(96, activation='relu', kernel_regularizer=regularizers.l2(L2))(h)
h    = layers.BatchNormalization()(h)
h    = layers.Dropout(0.5)(h)

outputs = layers.Dense(num_classes, activation='softmax',
                       kernel_regularizer=regularizers.l2(L2),
                       name='jogada')(h)

model = models.Model(inputs, outputs, name=f'BoxNet_v3_V8_{K}canais')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.KLDivergence(),
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name='accuracy'),
        tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc'),
        tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_acc'),
    ],
)
model.summary()


In [ ]:
# =========================================================================
# 3. TREINAMENTO
# =========================================================================
callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5, verbose=1),
]

history = model.fit(
    X_train, y_train,
    epochs=300,
    batch_size=256,
    validation_data=(X_val, y_val),
    sample_weight=sw,   # None se USE_SAMPLE_WEIGHT=False
    callbacks=callbacks,
    verbose=2,
)


In [ ]:
# =========================================================================
# 4. AVALIACAO TEXTUAL
# =========================================================================

# 4.1 Metricas gerais (treino / val / teste)
def avalia_conjunto(nome, X_, y_soft_):
    m = model.evaluate(X_, y_soft_, verbose=0, return_dict=True)
    return {'conjunto': nome, 'amostras': X_.shape[0],
            'kld_loss': m['loss'], 'top1_acc': m['accuracy'],
            'top3_acc': m['top3_acc'], 'top5_acc': m['top5_acc']}

resultados = pd.DataFrame([
    avalia_conjunto('Treino',    X_train, y_train),
    avalia_conjunto('Validacao', X_val,   y_val),
    avalia_conjunto('Teste',     X_test,  y_test),
])
print('=' * 70)
print('RESUMO DE DESEMPENHO (BoxNet v3 V8 — soft targets / KL Divergence)')
print('=' * 70)
print(resultados.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

gap_top1 = resultados.loc[0, 'top1_acc'] - resultados.loc[1, 'top1_acc']
gap_kld  = resultados.loc[1, 'kld_loss'] - resultados.loc[0, 'kld_loss']
print(f"\nGap top1 (Treino - Val): {gap_top1*100:+.2f} pp")
print(f"Gap KLD  (Val - Treino): {gap_kld:+.4f}")

ult = len(history.history['loss']) - 1
print(f"\nUltima epoca: {ult+1}")
print(f"  kld_loss  treino={history.history['loss'][ult]:.4f}  val={history.history['val_loss'][ult]:.4f}")
print(f"  top1_acc  treino={history.history['accuracy'][ult]:.4f}  val={history.history['val_accuracy'][ult]:.4f}")

# 4.2 Predicoes no conjunto de teste
y_pred_prob = model.predict(X_test, verbose=0)
y_pred_idx  = y_pred_prob.argmax(axis=1)

# 4.3 OMA — Optimal Move Accuracy (corrigido: aceita empates)
# pred_eh_otimo[i] = True se Q-value da jogada prevista == max(Q-values validos)
max_scores_test = S_test.max(axis=1, keepdims=True)
eh_otimo        = (S_test == max_scores_test) & (S_test > -1e8)
pred_eh_otimo   = eh_otimo[np.arange(len(y_pred_idx)), y_pred_idx]
oma_global      = pred_eh_otimo.mean()
media_equiv     = eh_otimo.sum(axis=1).mean()

print(f'\nOMA global (CNN in conjunto Minimax-otimo): {oma_global:.1%}')
print(f'Media de jogadas Minimax-equivalentes por estado: {media_equiv:.1f}')

# 4.4 Metricas por fase do jogo
fases = [
    (0,  11, 'Abertura (0-11)'),
    (12, 17, '1a Metade (12-17)'),
    (18, 23, '2a Metade (18-23)'),
    (24, 28, 'Fase Quente (24-28)'),
    (29, 31, 'Final (29-31)'),
]
print('\n' + '=' * 70)
print('TOP-1 / TOP-3 / TOP-5 / OMA POR FASE DO JOGO')
print('=' * 70)
print(f"  {'Fase':<24}  {'N':>6}  {'Top-1':>7}  {'Top-3':>7}  {'Top-5':>7}  {'OMA':>7}")
for lo, hi, nome in fases:
    mask = (tracos_test >= lo) & (tracos_test <= hi)
    if mask.sum() == 0:
        continue
    t1 = (y_pred_idx[mask] == y_test_idx[mask]).mean()
    argsorted = np.argsort(y_pred_prob[mask], axis=1)
    t3 = (argsorted[:, -3:] == y_test_idx[mask, np.newaxis]).any(axis=1).mean()
    t5 = (argsorted[:, -5:] == y_test_idx[mask, np.newaxis]).any(axis=1).mean()
    oma_f = pred_eh_otimo[mask].mean()
    print(f"  {nome:<24}  {mask.sum():>6}  {t1:>6.1%}  {t3:>6.1%}  {t5:>6.1%}  {oma_f:>6.1%}")

# 4.5 Classification report
BORDAS = {
    'H_0_1', 'H_0_3', 'H_0_5', 'H_8_1', 'H_8_3', 'H_8_5',
    'V_1_0', 'V_3_0', 'V_5_0', 'V_7_0', 'V_1_6', 'V_3_6', 'V_5_6', 'V_7_6',
}
print('\n' + '=' * 70)
print('CLASSIFICATION REPORT (conjunto de TESTE)')
print('=' * 70)
report = classification_report(
    y_test_idx, y_pred_idx,
    labels=list(range(num_classes)),
    target_names=[indice_para_rotulo[i] for i in range(num_classes)],
    digits=4, zero_division=0, output_dict=True,
)
print(f"  accuracy:      {report['accuracy']:.4f}")
print(f"  macro avg:     P={report['macro avg']['precision']:.4f}  "
      f"R={report['macro avg']['recall']:.4f}  F1={report['macro avg']['f1-score']:.4f}")
print(f"  weighted avg:  P={report['weighted avg']['precision']:.4f}  "
      f"R={report['weighted avg']['recall']:.4f}  F1={report['weighted avg']['f1-score']:.4f}")

por_classe = pd.DataFrame({
    rotulo: report[rotulo]
    for rotulo in [indice_para_rotulo[i] for i in range(num_classes)]
}).T
por_classe.index.name = 'jogada'
por_classe = por_classe.sort_values('f1-score', ascending=False)
por_classe['borda'] = por_classe.index.isin(BORDAS)
print("\nTop 10 jogadas com melhor F1:")
print(por_classe.head(10)[['precision', 'recall', 'f1-score', 'support', 'borda']]
      .to_string(float_format=lambda v: f"{v:.4f}"))
print("\nBottom 5 jogadas (onde o modelo mais erra):")
print(por_classe.tail(5)[['precision', 'recall', 'f1-score', 'support', 'borda']]
      .to_string(float_format=lambda v: f"{v:.4f}"))

# 4.6 Graficos de treino
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
axes[0].plot(history.history['accuracy'],     label='Treino')
axes[0].plot(history.history['val_accuracy'], label='Validacao')
axes[0].set_title('Top-1 Accuracy'); axes[0].legend(); axes[0].grid(True)
axes[1].plot(history.history['top3_acc'],     label='Treino')
axes[1].plot(history.history['val_top3_acc'], label='Validacao')
axes[1].set_title('Top-3 Accuracy'); axes[1].legend(); axes[1].grid(True)
axes[2].plot(history.history['loss'],     label='Treino (KLD)')
axes[2].plot(history.history['val_loss'], label='Validacao (KLD)')
axes[2].set_title('KL Divergence Loss'); axes[2].legend(); axes[2].grid(True)
plt.suptitle(f'BoxNet v3 V8 — {K} canais estruturais', fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
# =========================================================================
# 5. TABELA 1 — Presenca de canal por fase do jogo (conjunto de TESTE)
# Para cada fase e cada canal: quantas amostras tem ao menos uma celula = 1?
# Usa todos os 11 canais canonicos, independente de CANAIS_TREINAMENTO.
# =========================================================================

# canal_presente[i, k] = True se canais_test[i, :, :, k].any()
canal_presente = canais_test.reshape(len(canais_test), 4 * 3, 11).any(axis=1)  # (N_test, 11)

rows_t1 = []
for lo, hi, nome in fases:
    mask = (tracos_test >= lo) & (tracos_test <= hi)
    n_fase = int(mask.sum())
    row = {'Fase': nome, 'N': n_fase}
    for ki, nome_canal in enumerate(NOMES_CANAIS_REF):
        row[nome_canal] = int(canal_presente[mask, ki].sum())
    rows_t1.append(row)

df_t1 = pd.DataFrame(rows_t1).set_index('Fase')

print('=' * 90)
print('TABELA 1 — Amostras com canal presente por fase (N absoluto)')
print('=' * 90)
print(df_t1.to_string())

df_t1_pct = df_t1.copy()
for col in NOMES_CANAIS_REF:
    df_t1_pct[col] = (df_t1[col] / df_t1['N'] * 100).round(1)

print('\nPercentual de amostras com canal presente (%):')
print(df_t1_pct.to_string())


In [ ]:
# =========================================================================
# 6. TABELA 2 — Metricas por canal
# Para cada canal k: filtra amostras de TESTE onde o canal esta presente,
# e calcula Top-1, Top-3, Top-5, OMA nesse subconjunto.
# =========================================================================

rows_t2 = []
for ki, nome_canal in enumerate(NOMES_CANAIS_REF):
    mask = canal_presente[:, ki]
    n = int(mask.sum())
    if n == 0:
        rows_t2.append({'Canal': nome_canal, 'N': 0,
                        'Top-1': '—', 'Top-3': '—', 'Top-5': '—', 'OMA': '—'})
        continue
    y_pred_m  = y_pred_idx[mask]
    y_true_m  = y_test_idx[mask]
    y_pred_p  = y_pred_prob[mask]
    oma_m     = pred_eh_otimo[mask].mean()
    t1        = (y_pred_m == y_true_m).mean()
    argsorted = np.argsort(y_pred_p, axis=1)
    t3 = (argsorted[:, -3:] == y_true_m[:, np.newaxis]).any(axis=1).mean()
    t5 = (argsorted[:, -5:] == y_true_m[:, np.newaxis]).any(axis=1).mean()
    rows_t2.append({'Canal': nome_canal, 'N': n,
                    'Top-1': f'{t1:.1%}', 'Top-3': f'{t3:.1%}',
                    'Top-5': f'{t5:.1%}', 'OMA': f'{oma_m:.1%}'})

df_t2 = pd.DataFrame(rows_t2).set_index('Canal')
print('=' * 70)
print('TABELA 2 — Metricas por canal (subconjunto: amostras com canal presente)')
print('=' * 70)
print(df_t2.to_string())


In [ ]:
# =========================================================================
# 7. CORRELACAO CANAL x ERRO
# "Erro" = amostra onde pred_eh_otimo == False (CNN nao escolheu jogada otima).
# Para cada canal: taxa de presenca em erros vs media total.
# Delta positivo = canal sobrerrepresentado nos erros (possivel ponto fraco).
# =========================================================================

erros_mask = ~pred_eh_otimo
n_erros    = int(erros_mask.sum())
n_total    = len(erros_mask)
print('=' * 70)
print('CORRELACAO CANAL x ERRO (conjunto de TESTE)')
print(f'Erros (OMA=0): {n_erros} ({n_erros/n_total:.1%} do total de teste)')
print('=' * 70)

rows_corr = []
for ki, nome_canal in enumerate(NOMES_CANAIS_REF):
    taxa_total = float(canal_presente[:, ki].mean())
    taxa_erros = float(canal_presente[erros_mask, ki].mean()) if n_erros > 0 else 0.0
    delta      = (taxa_erros - taxa_total) * 100
    rows_corr.append({'Canal': nome_canal, 'taxa_total': taxa_total,
                      'taxa_erros': taxa_erros, 'delta_pp': delta})

df_corr = pd.DataFrame(rows_corr).sort_values('delta_pp', ascending=False)
df_corr_print = df_corr.copy()
df_corr_print['Total (%)']    = df_corr['taxa_total'].map(lambda v: f'{v:.1%}')
df_corr_print['Em Erros (%)'] = df_corr['taxa_erros'].map(lambda v: f'{v:.1%}')
df_corr_print['Delta (pp)']   = df_corr['delta_pp'].map(lambda v: f'{v:+.1f}')
print(df_corr_print[['Canal', 'Total (%)', 'Em Erros (%)', 'Delta (pp)']].to_string(index=False))

# Visualizacao
deltas = df_corr['delta_pp'].tolist()
nomes  = df_corr['Canal'].tolist()
colors = ['#d62728' if d > 0 else '#2ca02c' for d in deltas]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(nomes, deltas, color=colors, edgecolor='black', linewidth=0.5)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Delta (pp) — presenca em erros vs media do teste')
ax.set_title('Correlacao canal x erro  (vermelho = canal sobrerrepresentado nos erros)')
ax.set_xticklabels(nomes, rotation=45, ha='right', fontsize=9)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
# =========================================================================
# 8. VISUALIZACAO POR FASE (Top-1 / Top-3 / Top-5 / OMA — bar chart)
# =========================================================================
dados_fases = []
for lo, hi, nome in fases:
    mask = (tracos_test >= lo) & (tracos_test <= hi)
    if mask.sum() == 0:
        continue
    t1 = (y_pred_idx[mask] == y_test_idx[mask]).mean()
    argsorted = np.argsort(y_pred_prob[mask], axis=1)
    t3 = (argsorted[:, -3:] == y_test_idx[mask, np.newaxis]).any(axis=1).mean()
    t5 = (argsorted[:, -5:] == y_test_idx[mask, np.newaxis]).any(axis=1).mean()
    oma_f = pred_eh_otimo[mask].mean()
    fase_nome = nome.split(' ')[0]
    dados_fases.append({'Fase': fase_nome, 'Metrica': 'Top-1', 'Acuracia': t1})
    dados_fases.append({'Fase': fase_nome, 'Metrica': 'Top-3', 'Acuracia': t3})
    dados_fases.append({'Fase': fase_nome, 'Metrica': 'Top-5', 'Acuracia': t5})
    dados_fases.append({'Fase': fase_nome, 'Metrica': 'OMA',   'Acuracia': oma_f})

df_fases = pd.DataFrame(dados_fases)
plt.figure(figsize=(13, 6))
sns.barplot(data=df_fases, x='Fase', y='Acuracia', hue='Metrica', palette='viridis')
plt.title(f'Performance por Fase — BoxNet v3 V8 ({K} canais)')
plt.ylim(0, 1.1)
plt.ylabel('Taxa de Acerto')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


In [ ]:
# =========================================================================
# 9. EXPORTACAO PARA TENSORFLOW LITE
# =========================================================================
import shutil

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

nome_arquivo = f'pontinhos_pequeno_profundidade_11_v8_{K}canais.tflite'

# 1) Salva localmente na instancia Colab
with open(nome_arquivo, 'wb') as f:
    f.write(tflite_model)
print(f'Modelo salvo localmente: {nome_arquivo} ({len(tflite_model)/1024:.1f} KB)')

# 2) Copia para o Google Drive
drive_path = os.path.join(DRIVE_OUTPUT_DIR, nome_arquivo)
shutil.copy(nome_arquivo, drive_path)
print(f'Modelo salvo no Drive: {drive_path}')

# 3) Download direto (funciona se a sessao Colab ainda estiver ativa)
try:
    from google.colab import files
    files.download(nome_arquivo)
except Exception:
    print('Download automatico nao disponivel — use o arquivo salvo no Drive.')


In [ ]:
# =========================================================================
# 10. EXPORTAR OUTPUTS DO NOTEBOOK PARA MARKDOWN
# =========================================================================
import json, os, re, shutil
from datetime import datetime


def _texto_do_output(output: dict) -> str:
    ot = output.get('output_type', '')
    if ot == 'stream':
        text = output.get('text', [])
    elif ot in ('execute_result', 'display_data'):
        text = output.get('data', {}).get('text/plain', [])
    else:
        return ''
    return ''.join(text).strip() if isinstance(text, list) else str(text).strip()


def _obter_notebook() -> dict | None:
    try:
        from google.colab import _message
        return _message.blocking_request('get_ipynb', request='', timeout_sec=10)
    except Exception:
        pass
    import glob
    for path in glob.glob('/content/*.ipynb'):
        try:
            with open(path, encoding='utf-8') as f:
                return json.load(f)
        except Exception:
            pass
    return None


def salvar_outputs_como_md(destino: str = DRIVE_OUTPUT_DIR) -> str | None:
    nb = _obter_notebook()
    if nb is None:
        print('Nao foi possivel obter o notebook. Salve o arquivo e tente novamente.')
        return None

    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    nome = f'resultados_v8_{K}canais_{ts}.md'

    linhas = [
        f'# Resultados — BoxNet V8 ({K} canais)\n\n',
        f'**Data**: {datetime.now().strftime("%Y-%m-%d %H:%M")}  \n',
        f'**Canais**: `{_canais_alvo}`  \n',
        f'**PASTA_NPZ**: `{PASTA_NPZ}`\n\n---\n\n',
    ]

    for cell in nb.get('cells', []):
        if cell.get('cell_type') != 'code':
            continue

        outputs = cell.get('outputs', [])
        if not outputs:
            continue

        for out in outputs:
            texto = _texto_do_output(out)
            if texto:
                linhas.append(f'```\n{texto}\n```\n\n')
            elif 'image/png' in out.get('data', {}):
                linhas.append('*[figura — ver notebook]*\n\n')

    md = ''.join(linhas)
    with open(nome, 'w', encoding='utf-8') as f:
        f.write(md)
    print(f'Salvo localmente: {nome} ({len(md.encode())/1024:.1f} KB)')

    drive_path = os.path.join(destino, nome)
    try:
        shutil.copy(nome, drive_path)
        print(f'Copiado para o Drive: {drive_path}')
    except Exception as e:
        print(f'Drive indisponivel ({e})')

    try:
        from google.colab import files
        files.download(nome)
    except Exception:
        pass

    return nome


salvar_outputs_como_md()